In [4]:
import altair as alt
alt.data_transformers.disable_max_rows()
import pandas as pd
import numpy as np

In [5]:
PRESETS = {
    "education": {
        "color": "#9C0505",
        "title_prefix": "Education",
        "default_value_col": "value",
        "default_category_col": "region",
        "notes": "e.g., years of schooling, attainment %, test scores"
    },
    "health": {
        "color": "#059C19",
        "title_prefix": "Health",
        "default_value_col": "value",
        "default_category_col": "region",
        "notes": "e.g., life expectancy, mortality rates, health coverage"
    }
}

In [98]:

def create_country_bar_chart(df, title_text="", axis_title="", color_t =""):
    """
    Creates a bar chart of education attainment by country.
    """
    chart = alt.Chart(df).mark_bar(color=color_t).encode(
        x=alt.X('Reference area:N', sort='-y', title='Country'),
        y=alt.Y('OBS_VALUE:Q', title=axis_title,
                axis=alt.Axis(grid=True, gridColor="#dbdbdb", gridOpacity=0.3)),
        tooltip=['Reference area', 'OBS_VALUE']
    ).properties(
        title=alt.TitleParams(
            text=title_text,
            anchor='start',
            offset=10,
            dx=20
        ),
        width=800,
        height=400
    ).resolve_legend(
        color='independent',
        shape='independent'
    )
    return chart





def create_regional_gap_chart(chart_df, country_order):
    """
    Creates a chart showing the gap between best and worst regions for each country.
    """
    gap_bars = alt.Chart(chart_df[chart_df['Type'].isin(['Worst Region', 'Best Region'])]).mark_rule(
        strokeWidth=8,
        stroke='lightgray',
        opacity=0.4
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country'),
        y=alt.Y('min(Value):Q', axis=alt.Axis(gridColor="#f3f1f1e1")),
        y2=alt.Y2('max(Value):Q'),
        tooltip=[
            alt.Tooltip('Country:N', title='Country'),
            alt.Tooltip('min(Value):Q', title='Worst Region Deviation'),
            alt.Tooltip('max(Value):Q', title='Best Region Deviation')
        ]
    )
    return gap_bars

def create_deviation_points_chart(chart_df, country_order, average_color):
    """
    Creates a points chart for deviations from regional median.
    """
    points = alt.Chart(chart_df).mark_point(
        size=100,
        filled=True
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country'),
        y=alt.Y('Value:Q', title='Deviation from Regional Median (percentage points)'),
        shape=alt.Shape('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['triangle-up', 'square', 'triangle-down']),
            legend=alt.Legend(orient='top'),
            title=''),
        color=alt.Color('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['#3C3F3C', average_color, "#3C3F3C"]),
            legend=alt.Legend(orient='top'),
            title=''),
        tooltip=['Country', 'Type', 'Value']
    )
    return points

def create_zero_line():
    """
    Creates a horizontal dashed line at y=0 for baseline.
    """
    zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
        strokeDash=[5, 5],
        stroke='black',
        strokeWidth=1.5,
        opacity=0.6
    ).encode(
        y=alt.Y('y:Q')
    )
    return zero_line

def create_source_text():
    """
    Creates a source text annotation for charts.
    """
    source_text = alt.Chart(pd.DataFrame({'source': ['Source: OECD']})).mark_text(
        align='left',
        fontSize=10,
        color='gray',
        dx=-400
    ).encode(
        text='source:N'
    ).properties(
        width=800,
        height=20
    )
    return source_text

def create_regional_variation_chart(chart_df, country_order, average_color, chart_title= ""):
    """
    Combines gap bars, zero line, and deviation points into a single chart.
    """
    gap_bars = create_regional_gap_chart(chart_df, country_order)
    zero_line = create_zero_line()
    points = create_deviation_points_chart(chart_df, country_order, average_color)
    main_chart = (gap_bars + zero_line + points).properties(
        title=alt.TitleParams(
            text=chart_title,
            anchor='start',
            offset=10,
            dx=20
        ),
        width=800,
        height=400
    ).resolve_legend(
        color='independent',
        shape='independent'
    )
    source_text = create_source_text()
    chart = alt.vconcat(
        main_chart,
        source_text,
        spacing=10
    ).resolve_scale(
        color='independent',
        shape='independent'
    )
    return chart

    # scatter plot of GDP 
def scatter_gdp(merged_data, chart_title, y_title, chart_color, y_min, jump):

    chart = alt.Chart(merged_data).mark_circle(size=250, color=chart_color).encode(
    x=alt.X(
        'log_2023:Q',
        title="GDP per Capita (2023 - log scale)",
        scale=alt.Scale(domain=[9, merged_data['log_2023'].max()+0.5]),
        axis=alt.Axis(
            values=list(range(9, int(np.ceil(merged_data['log_2023'].max())) + 1)),
            tickMinStep=1
        )
    ),
    y=alt.Y(
        'OBS_VALUE:Q', 
        title= y_title,
        scale=alt.Scale(domain=[y_min, merged_data['OBS_VALUE'].max()+0.5]),
        axis=alt.Axis(
            grid=False, gridColor="#fffdfd", gridOpacity=0.3,
            values=list(range(y_min, int(np.ceil(merged_data['OBS_VALUE'].max())) + 1,jump)),
            #tickMinStep=1
        )
    ),
    tooltip=['Reference area', 'OBS_VALUE', '2023']
    ).properties(
    title=chart_title,
    width=400,
    height=400
    )
    return chart

# Health indicators

In [7]:
health_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_regional_data.csv")
countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_countries_complete.csv")

In [8]:
gdp_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\GDP.csv")

In [ ]:
def filter_by_sex(df, sex='Total'):
    """Filter dataframe by sex."""
    return df[df['Sex'] == sex]

def get_regional_stats(df, min_regions=5):
    """Group by country and calculate min, max, count for OBS_VALUE. Filter by min_regions."""
    stats = df.groupby('COUNTRY')['OBS_VALUE'].agg(['min', 'max', 'count']).reset_index()
    stats_filtered = stats[stats['count'] >= min_regions]
    return stats_filtered

def merge_country_names(stats_df, names_df):
    """Merge stats with country names."""
    return stats_df.merge(names_df[['COUNTRY', 'Country']].drop_duplicates(), on='COUNTRY')

def create_minmax_points(stats_df):
    """Create separate dataframes for min and max points and combine them."""
    min_points = stats_df[['Country', 'min']].rename(columns={'min': 'OBS_VALUE'})
    min_points['Type'] = 'Regional Min'
    max_points = stats_df[['Country', 'max']].rename(columns={'max': 'OBS_VALUE'})
    max_points['Type'] = 'Regional Max'
    minmax_points = pd.concat([min_points, max_points], ignore_index=True)
    minmax_points['Reference area'] = minmax_points['Country']
    return minmax_points

def filter_countries_with_regions(countries_df, valid_countries):
    """Filter country data to only include countries with enough regions."""
    return countries_df[countries_df['Reference area'].isin(valid_countries)]

def get_country_sort_order(countries_df):
    """Get country sorting order by national average."""
    return countries_df.sort_values('OBS_VALUE', ascending=False)['Reference area'].tolist()

def create_national_data(countries_df):
    """Prepare national average data for chart."""
    national_data = countries_df.copy()
    national_data['Type'] = 'National Average'
    national_data['Reference area'] = national_data['Reference area']
    return national_data[['Reference area', 'OBS_VALUE', 'Type']]

def combine_all_points(minmax_points, national_data):
    """Combine min/max points and national average data."""
    return pd.concat([minmax_points, national_data], ignore_index=True)

def merge_gdp_with_national_data(national_data, gdp_2023):
    """
    Merge GDP data with national health data using country names.
    Adds log_2023 as the natural log of the 2023 GDP value.
    Returns a DataFrame with GDP and health columns.
    """
    merged = national_data.merge(
        gdp_2023,
        left_on='Reference area',
        right_on='Country Name',
        how='left'
    )
    merged['log_2023'] = np.log(merged['2023'])
    return merged

# Data Preparation
regional_data_total = filter_by_sex(health_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(countries_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, national_data)

In [10]:
def compute_normalized_deviations(regional_stats_filtered, regional_data_total, countries_filtered):
    """
    For each country, compute deviations from the regional median for national average, min, and max.
    Returns a sorted DataFrame with these values and the gap.
    """
    normalized_data = []
    for country in regional_stats_filtered['Country'].unique():
        # Get all regional values for this country
        country_regional_data = regional_data_total[regional_data_total['Country'] == country]
        regional_median = country_regional_data['OBS_VALUE'].median()

        filtered = countries_filtered[countries_filtered['Reference area'] == country]
        if not filtered.empty:
            national_avg = filtered['OBS_VALUE'].iloc[0]
        else:
            national_avg = float('nan')

        # Get regional min/max for this country
        country_stats = regional_stats_filtered[regional_stats_filtered['Country'] == country]
        min_val = country_stats['min'].iloc[0]
        max_val = country_stats['max'].iloc[0]

        # Calculate deviations from regional median
        national_deviation = national_avg - regional_median
        min_deviation = min_val - regional_median
        max_deviation = max_val - regional_median
        gap = max_val - min_val

        normalized_data.append({
            'Country': country,
            'Regional_Median': 0,
            'National_Deviation': national_deviation,
            'Min_Deviation': min_deviation,
            'Max_Deviation': max_deviation,
            'Gap': gap
        })

    norm_df = pd.DataFrame(normalized_data)
    norm_df_sorted = norm_df.sort_values('Gap', ascending=False)
    return norm_df_sorted

def build_chart_data(norm_df_sorted):
    """
    Create chart data for Altair from normalized deviations DataFrame.
    Returns chart_df and country_order.
    """
    chart_data = []
    for _, row in norm_df_sorted.iterrows():
        chart_data.extend([
            {'Country': row['Country'], 'Value': row['Min_Deviation'], 'Type': 'Worst Region'},
            {'Country': row['Country'], 'Value': 0, 'Type': 'Regional Median'},
            {'Country': row['Country'], 'Value': row['National_Deviation'], 'Type': 'National Average'},
            {'Country': row['Country'], 'Value': row['Max_Deviation'], 'Type': 'Best Region'}
        ])
    chart_df = pd.DataFrame(chart_data)
    country_order = norm_df_sorted['Country'].tolist()
    return chart_df, country_order

# 
norm_df_sorted = compute_normalized_deviations(regional_stats_filtered, regional_data_total, countries_filtered)
chart_df, country_order = build_chart_data(norm_df_sorted)

In [11]:
chart_life = create_country_bar_chart(national_data, title_text="Life Expectancy by Country (Circa 2024)", axis_title="Life Expectancy (years)", color_t=PRESETS["health"]["color"])
chart_life.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\bar_health_life_expectancy_by_country.png")
chart_life

alt.Chart(...)

In [12]:
#
chart_life_ine = create_regional_variation_chart(chart_df, country_order, average_color="#059C19", chart_title="Regional Life Expectancy Variation by Country (Circa 2024)")
chart_life_ine.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_health_regional_life_expectancy_variation.png")
chart_life_ine

alt.VConcatChart(...)

In [13]:
mortality_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_regional_data.csv")
mortality_countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_countries_complete.csv")

In [14]:
# transform to survival rate for better visualization
mortality_regional_data['survival_rate'] = (1- (mortality_regional_data['OBS_VALUE'] / 1000))*100
mortality_countries_with_additions['survival_rate'] = (1- (mortality_countries_with_additions['OBS_VALUE'] / 1000))*100

mortality_regional_data['mortality_rate'] = mortality_regional_data['OBS_VALUE']
mortality_countries_with_additions['mortality_rate'] = mortality_countries_with_additions['OBS_VALUE']

mortality_regional_data.drop(columns=['OBS_VALUE'], inplace=True)
mortality_countries_with_additions.drop(columns=['OBS_VALUE'], inplace=True)

mortality_regional_data['OBS_VALUE'] = mortality_regional_data['survival_rate']
mortality_countries_with_additions['OBS_VALUE'] = mortality_countries_with_additions['survival_rate']

#rename OBS_VALUE to survival_rate for clarity
#mortality_regional_data = mortality_regional_data.rename(columns={'survival_rate': 'OBS_VALUE'})
#mortality_countries_with_additions = mortality_countries_with_additions.rename(columns={'survival_rate': 'OBS_VALUE'})


In [15]:
regional_data_total = filter_by_sex(mortality_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(mortality_countries_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
national_data_survival = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, national_data_survival)

norm_df_sorted = compute_normalized_deviations(regional_stats_filtered, regional_data_total, countries_filtered)
chart_df_survival, country_order = build_chart_data(norm_df_sorted)

In [16]:
create_country_bar_chart(national_data_survival, title_text="Child Survival rate by Country (Circa 2024)", axis_title="Child Survival Rate (%)", color_t="#059C19")
#  save chart as png
chart = create_country_bar_chart(national_data_survival, title_text="Child Survival rate by Country (Circa 2024)", axis_title="Child Survival Rate (%)", color_t="#059C19")
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\bar_health_child_survival_rate_by_country.png")
chart

alt.Chart(...)

In [17]:
# save this graph as png
chart = create_regional_variation_chart(chart_df_survival, country_order, average_color="#059C19", chart_title="Regional Child Survival Rate Variation by Country (Circa 2024)")
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_health_regional_child_survival_rate_variation.png")
chart

alt.VConcatChart(...)

In [18]:
gdp_2023 = gdp_df[['Country Name','Country Code',  '2023']]
merged_data = merge_gdp_with_national_data(national_data_survival, gdp_2023)

In [19]:
# save this graph as png
chart_gdp1 = scatter_gdp(merged_data, 'Child survival rate vs GDP per Capita (2023 - log)', 'Child Survival Rate (%)', '#059C19', 96, 2)
chart_gdp1.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_health_child_survival_vs_gdp.png")
chart_gdp1

alt.Chart(...)

In [20]:
# Usage example:
merged_data2 = merge_gdp_with_national_data(national_data, gdp_2023)
merged_data2.head()

,Reference area,OBS_VALUE,Type,Country Name,Country Code,2023,log_2023
0,Australia,83.00,National Average,Australia,AUS,60461.15640,11.009756
1,Austria,81.90,National Average,Austria,AUT,64394.09052,11.072777
2,Bulgaria,75.80,National Average,Bulgaria,BGR,33141.84442,10.408552
3,Canada,81.34,National Average,Canada,CAN,57517.44316,10.959844
4,Switzerland,84.30,National Average,Switzerland,CHE,82302.04918,11.318151


In [21]:
chart_gdp2 = scatter_gdp(merged_data2, 'Life expectancy vs GDP per Capita (2023 - log)', 'Life Expectancy (years)', '#059C19', 65, 5)
chart_gdp2.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_health_life_expectancy_vs_gdp.png")

# Education indicators

Education Attainment: population aged 25 - 64 with terciary education

In [22]:
attainment_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_regional_data.csv")
attainment_countries_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_countries_complete.csv")

In [23]:
regional_stats_filtered = get_regional_stats(attainment_regional_data, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, attainment_regional_data)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(attainment_countries_with_additions, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
attainment_national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, attainment_national_data)

In [24]:
attainment_norm_df_sorted = compute_normalized_deviations(regional_stats_filtered, attainment_regional_data, countries_filtered)
attainment_chart_df, country_order = build_chart_data(attainment_norm_df_sorted)

In [25]:
chart = create_country_bar_chart(attainment_national_data, title_text="Education Attainment of Population Aged 25 - 64 by Country (Circa 2024)", axis_title="Population aged 25 - 64 with tertiary education (%)", color_t=PRESETS["education"]["color"])
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\bar_education_attainment_by_country.png")
chart

alt.Chart(...)

In [26]:
#
chart = create_regional_variation_chart(attainment_chart_df, country_order, average_color=PRESETS["education"]["color"], chart_title="Education Attainment Regional Variation: Deviation from Regional Median (Circa 2024)")
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_education_regional_attainment_variation.png")

In [27]:
attainment_merged_data = merge_gdp_with_national_data(attainment_national_data, gdp_2023)


In [28]:
chart_gdp3 = scatter_gdp(attainment_merged_data, 'Education Attainment vs GDP per Capita (2023 - log)', 'Education Attainment (%)', PRESETS["education"]["color"], 15,5)
chart_gdp3.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_education_attainment_vs_gdp.png")

School enrollment of population aged 15 - 19

In [29]:
enrollment_regional_data = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_regional_data.csv")
enrollment_with_additions = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_countries_complete.csv")

In [30]:
regional_data_total = filter_by_sex(enrollment_regional_data, sex='Total')
countries_with_additions_total = filter_by_sex(enrollment_with_additions, sex='Total')

regional_stats_filtered = get_regional_stats(regional_data_total, min_regions=5)
regional_stats_filtered = merge_country_names(regional_stats_filtered, regional_data_total)

minmax_points = create_minmax_points(regional_stats_filtered)
countries_with_5plus_regions = regional_stats_filtered['Country'].unique()
countries_filtered = filter_countries_with_regions(countries_with_additions_total, countries_with_5plus_regions)
country_sort_order = get_country_sort_order(countries_filtered)
enrollment_national_data = create_national_data(countries_filtered)
all_points_data = combine_all_points(minmax_points, enrollment_national_data)

norm_df_sorted = compute_normalized_deviations(regional_stats_filtered, regional_data_total, countries_filtered)
enrollment_chart_df, country_order = build_chart_data(norm_df_sorted)

In [31]:
regional_stats_filtered[regional_stats_filtered['Country'] == 'Japan']

,COUNTRY,min,max,count,Country
20,JPN,89.3,714.9,10,Japan


In [32]:
chart = create_country_bar_chart(enrollment_national_data, title_text="School Enrollment of Population Aged 15 - 19 by Country (Circa 2024)", axis_title="Enrollment Rate (%)", color_t=PRESETS["education"]["color"])
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\bar_education_enrollment_by_country.png")

In [33]:
enrollment_chart_df[enrollment_chart_df['Country'] == 'Colombia']

,Country,Value,Type
12,Colombia,-25.8,Worst Region
13,Colombia,0.0,Regional Median
14,Colombia,3.4,National Average
15,Colombia,16.7,Best Region


In [34]:
enrollment_chart_df = enrollment_chart_df[enrollment_chart_df['Country'] != 'Japan'] # japan data is looking weird

In [35]:
#
chart = create_regional_variation_chart(enrollment_chart_df, country_order, average_color=PRESETS["education"]["color"], chart_title="School Enrollment Regional Variation: Deviation from Regional Median (Circa 2024)")
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\shape_education_regional_enrollment_variation.png")

In [36]:
enrollment_merged_data = merge_gdp_with_national_data(enrollment_national_data, gdp_2023)

In [37]:
chart_gdp4 = scatter_gdp(enrollment_merged_data, 'School Enrollment vs GDP per Capita (2023 - log)', 'School Enrollment (%)', PRESETS["education"]["color"], 60, 5)
chart_gdp4.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_education_school_enrollment_vs_gdp.png")
chart_gdp4

alt.Chart(...)

# Building a HCI

In [38]:
def _norm_clip(series, lo=0.0, hi=1.0):
    s = pd.to_numeric(series, errors="coerce")
    return s.clip(lower=lo, upper=hi)

def _life_index(le_years: pd.Series, lo=20.0, hi=85.0):
    return _norm_clip((le_years - lo) / (hi - lo), 0.0, 1.0)

def _pct_index(pct: pd.Series):
    return _norm_clip(pct / 100.0, 0.0, 1.0)

def _gmean(a: pd.Series, b: pd.Series):
    return np.sqrt(a * b)

def build_national_hci(Life: pd.DataFrame,
                       survival: pd.DataFrame,
                       attainment: pd.DataFrame,
                       enrollment: pd.DataFrame,
                       area_col: str = "Reference area",
                       value_col: str = "OBS_VALUE"):
    
    L = Life[[area_col, value_col]].rename(columns={value_col: "LE"})
    S = survival[[area_col, value_col]].rename(columns={value_col: "Survival"})
    A = attainment[[area_col, value_col]].rename(columns={value_col: "Attainment"})
    E = enrollment[[area_col, value_col]].rename(columns={value_col: "Enrollment"})
    df = L.merge(S, on=area_col, how="inner").merge(A, on=area_col, how="inner").merge(E, on=area_col, how="inner")
    df["LE_idx"] = _life_index(df["LE"])
    df["Survival_idx"] = _pct_index(df["Survival"])
    df["Health_idx"] = _gmean(df["LE_idx"], df["Survival_idx"])
    df["Attain_idx"] = _pct_index(df["Attainment"])
    df["Enroll_idx"] = _pct_index(df["Enrollment"])
    df["Educ_idx"] = _gmean(df["Attain_idx"], df["Enroll_idx"])
    df["HCI_composite"] = _gmean(df["Health_idx"], df["Educ_idx"])
    cols = [area_col, "LE", "Survival", "Attainment", "Enrollment",
            "LE_idx", "Survival_idx", "Health_idx", "Attain_idx", "Enroll_idx", "Educ_idx", "HCI_composite"]
    return df[cols].sort_values("HCI_composite", ascending=False).reset_index(drop=True)

In [39]:
Life = national_data
survival = national_data_survival
attainment = attainment_national_data
enrollment = enrollment_national_data

In [40]:
hci_national = build_national_hci(Life, survival, attainment, enrollment)
hci_national.head(30)

,Reference area,LE,Survival,Attainment,Enrollment,LE_idx,Survival_idx,Health_idx,Attain_idx,Enroll_idx,Educ_idx,HCI_composite
0,Canada,81.34,99.560000,63.000000,83.9,0.943692,0.995600,0.969299,0.630000,0.839,0.727028,0.839469
1,Sweden,83.40,99.790000,48.500000,87.5,0.975385,0.997900,0.986578,0.485000,0.875,0.651441,0.801684
2,Norway,83.10,99.800000,47.800000,88.2,0.970769,0.998000,0.984290,0.478000,0.882,0.649304,0.799440
3,Netherlands,81.90,99.640000,44.700000,92.3,0.952308,0.996400,0.974104,0.447000,0.923,0.642325,0.791007
4,Switzerland,84.30,99.670000,44.600000,85.9,0.989231,0.996700,0.992958,0.446000,0.859,0.618962,0.783967
5,United Kingdom,80.70,99.610000,48.675000,80.9,0.933846,0.996100,0.964471,0.486750,0.809,0.627520,0.777962
6,United States,77.00,99.420000,47.011765,86.7,0.876923,0.994200,0.933722,0.470118,0.867,0.638429,0.772085
7,Finland,81.60,99.820000,42.700000,87.9,0.947692,0.998200,0.972618,0.427000,0.879,0.612644,0.771926
8,France,83.00,99.600000,41.600000,87.6,0.969231,0.996000,0.982524,0.416000,0.876,0.603669,0.770142
9,Spain,83.77,99.740000,40.700000,87.8,0.981077,0.997400,0.989205,0.407000,0.878,0.597784,0.768981


In [41]:
def add_contribution_shares(df, comp_cols=("LE_idx","Survival_idx","Attain_idx","Enroll_idx"), hci_col="HCI_composite"):
    """
    Allocate HCI_composite proportionally to components so contributions sum to HCI_composite.

    """
    eps = 1e-9
    dfc = df.copy()
    dfc["HCI_arith"] = dfc[list(comp_cols)].mean(axis=1)
    comp_sum = dfc[list(comp_cols)].sum(axis=1).replace(0, eps)
    for c in comp_cols:
        dfc[f"Contrib_{c}"] = (dfc[c] / comp_sum) * dfc[hci_col]
    return dfc

def plot_hci_stack(df, area_col="Reference area", top=10, weighted=True, colors=None, use_contributions=False, border_color="lightgray", border_width=0.9):
    """
    Simplified stacked HCI chart using the provided parameters.
    """
    comp_cols = ["LE_idx", "Survival_idx", "Attain_idx", "Enroll_idx"]

    # prepare plot dataframe and variable names
    if use_contributions:
        df_plot = add_contribution_shares(df, comp_cols=comp_cols)
        plot_vars = [f"Contrib_{c}" for c in comp_cols]
    else:
        df_plot = df.copy()
        plot_vars = comp_cols

    long = df_plot.melt(id_vars=[area_col, "HCI_composite"], value_vars=plot_vars, var_name="Component", value_name="Index")
    if use_contributions:
        long["Component"] = long["Component"].str.replace(r"^Contrib_", "", regex=True)
    if weighted and not use_contributions:
        long["Index"] = long["Index"] * 0.25

    # top countries selection
    top_countries = df.nlargest(top, "HCI_composite")[area_col].tolist()
    plot_df = long[long[area_col].isin(top_countries)]

    # color encoding (handles None, dict, or list)
    if isinstance(colors, dict):
        # assume keys match the Component values (LE_idx, etc.)
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=list(colors.keys()), range=list(colors.values())),
                              legend=alt.Legend(title="Sub-index", orient="top"))
    elif isinstance(colors, list):
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=comp_cols, range=colors),
                              legend=alt.Legend(title="Sub-index", orient="top"))
    else:
        color_enc = alt.Color("Component:N", legend=alt.Legend(title="Sub-index", orient="top"))

    chart = (
        alt.Chart(plot_df)
        .mark_bar(stroke=border_color, strokeWidth=border_width)
        .encode(
            x=alt.X(f"{area_col}:N",
                    sort=alt.SortField(field="HCI_composite", order="descending"),
                    title="Country"),
            y=alt.Y("sum(Index):Q", stack="zero", title="HCI composite",
                    axis=alt.Axis(grid=True, gridColor="#dbdbdb", gridOpacity=0.3)),
            color=color_enc,
            tooltip=[area_col, "Component", "Index", alt.Tooltip("HCI_composite:Q", title="HCI")]
        )
        .properties(width=20 * top, height=300, title=f"Stacked contribution of HCI components (Top {top})")
    )

    return chart

In [42]:
def plot_hci_stack(df, area_col="Reference area", top=10, weighted=True, colors=None, use_contributions=False, border_color="lightgray", border_width=0.9):
    """
    Simplified stacked HCI chart using the provided parameters.
    """
    comp_cols = ["LE_idx", "Survival_idx", "Attain_idx", "Enroll_idx"]

    # prepare plot dataframe and variable names
    if use_contributions:
        df_plot = add_contribution_shares(df, comp_cols=comp_cols)
        plot_vars = [f"Contrib_{c}" for c in comp_cols]
    else:
        df_plot = df.copy()
        plot_vars = comp_cols

    long = df_plot.melt(id_vars=[area_col, "HCI_composite"], value_vars=plot_vars, var_name="Component", value_name="Index")
    if use_contributions:
        long["Component"] = long["Component"].str.replace(r"^Contrib_", "", regex=True)
    if weighted and not use_contributions:
        long["Index"] = long["Index"] * 0.25

    # --- map internal component keys to friendly labels for legend/display ---
    label_map = {
        "LE_idx": "Life expectancy",
        "Survival_idx": "Survival rate",
        "Attain_idx": "Educational attainment",
        "Enroll_idx": "Enrollment rate"
    }
    long["Component"] = long["Component"].map(lambda k: label_map.get(k, k))

    # top countries selection
    top_countries = df.nlargest(top, "HCI_composite")[area_col].tolist()
    plot_df = long[long[area_col].isin(top_countries)]

    # color encoding (handles None, dict, or list) -- use mapped labels as domain
    if isinstance(colors, dict):
        mapped_domain = [label_map.get(k, k) for k in list(colors.keys())]
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=mapped_domain, range=list(colors.values())),
                              legend=alt.Legend(title="", orient="top"))
    elif isinstance(colors, list):
        mapped_domain = [label_map[c] for c in comp_cols]
        color_enc = alt.Color("Component:N",
                              scale=alt.Scale(domain=mapped_domain, range=colors),
                              legend=alt.Legend(title="", orient="top"))
    else:
        color_enc = alt.Color("Component:N", legend=alt.Legend(title="", orient="top"))

    chart = (
        alt.Chart(plot_df)
        .mark_bar(stroke=border_color, strokeWidth=border_width)
        .encode(
            x=alt.X(f"{area_col}:N",
                    sort=alt.SortField(field="HCI_composite", order="descending"),
                    title="Country"),
            y=alt.Y("sum(Index):Q", stack="zero", title="HCI composite",
                    axis=alt.Axis(grid=True, gridColor="#dbdbdb", gridOpacity=0.3)),
            color=color_enc,
            tooltip=[area_col, "Component", "Index", alt.Tooltip("HCI_composite:Q", title="HCI")]
        )
        .properties(width=20 * top, height=300, title=f"Stacked contribution of HCI components (available countries CIRCA 2024)")
    )

    return chart

In [50]:
custom_colors = {
    "LE_idx": "#4997536E",       # blue
    "Survival_idx": "#059C19", # green
    "Attain_idx": "#9C05055D",   # orange
    "Enroll_idx": "#9C0505"    # red
}

plot_hci_stack(hci_national, top=30, weighted=False, colors=custom_colors, use_contributions=True)

alt.Chart(...)

In [438]:
chart = plot_hci_stack(hci_national, top=30, weighted=False, colors=custom_colors, use_contributions=True)
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\stacked_hci_components_by_country.png")

In [380]:
# save this graph as png
chart = plot_hci_stack(hci_national, top=30, weighted=False, colors=custom_colors, use_contributions=True)
chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\hci_stacked_chart.png")

In [378]:
hci_national["OBS_VALUE"] = hci_national["HCI_composite"]
print(type(hci_national)) 
create_country_bar_chart(hci_national, title_text="Human Capital Index, National (Circa 2024)", axis_title="HCI", color_t="grey")

<class 'pandas.core.frame.DataFrame'>


alt.Chart(...)

In [65]:
# layered chart on the country_year_hci data
hci_change_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\hci_country_year.csv")
hci_regional_change_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\hci_region_year.csv")

In [62]:
hci_regional_change_df.shape

(986, 16)

In [115]:
def merge_country_names(stats_df, names_df):
    """Merge stats with country names robustly."""
    s = stats_df.copy()
    n = names_df.copy()

    # if stats already has a 'Country' column, nothing to do
    if 'Country' in s.columns:
        return s

    # common case: stats has 'COUNTRY' and names_df has 'Country' -> join them
    if 'COUNTRY' in s.columns and 'Country' in n.columns:
        return s.merge(n[['Country', 'OBS_VALUE']].drop_duplicates(), left_on='COUNTRY', right_on='Country', how='left')

    # if both have 'COUNTRY' (rare) join on that and keep the printable 'Country' if present
    if 'COUNTRY' in s.columns and 'COUNTRY' in n.columns:
        return s.merge(n[['COUNTRY', 'Country', 'OBS_VALUE']].drop_duplicates(), on='COUNTRY', how='left')

    # fallback: create a 'Country' column from 'COUNTRY' if present
    if 'COUNTRY' in s.columns:
        s['Country'] = s['COUNTRY']
        return s

    raise KeyError("merge_country_names: couldn't find suitable country identifier in stats_df or names_df")

In [144]:
# changing names

hci_change_df.rename(columns={'HCI_composite': 'OBS_VALUE'}, inplace=True)
hci_regional_change_df.rename(columns={'HCI_composite': 'OBS_VALUE'}, inplace=True)

hci_regional_change_df_max = hci_regional_change_df[hci_regional_change_df['which'] == 'max']
regional_data_total_max = hci_change_df[hci_change_df['which'] == 'max']

hci_regional_change_df_min = hci_regional_change_df[hci_regional_change_df['which'] == 'min']
regional_data_total_min = hci_change_df[hci_change_df['which'] == 'min']

regional_stats_filtered_max = get_regional_stats(hci_regional_change_df_max, min_regions=5)

regional_stats_filtered_min = get_regional_stats(hci_regional_change_df_min, min_regions=5)

# circa 2024 - max circa 2010 min
regional_stats_filtered_max = merge_country_names(regional_stats_filtered_max, regional_data_total_max)
regional_stats_filtered_min = merge_country_names(regional_stats_filtered_min, regional_data_total_min)

print(regional_stats_filtered_max.head(15))
print("\n")
print(regional_stats_filtered_min.head(15))

      COUNTRY       min       max  count    Country  OBS_VALUE
0   Australia  0.715821  0.882931      8  Australia   0.786343
1     Austria  0.678598  0.783333      9    Austria   0.713165
2    Bulgaria  0.572023  0.752261      6   Bulgaria   0.639161
3      Canada  0.660886  0.849759     12     Canada   0.787449
4       Chile  0.651181  0.740453     16      Chile   0.688405
5    Colombia  0.491184  0.712793     24   Colombia   0.560072
6     Czechia  0.575482  0.832470      8    Czechia   0.676822
7     Denmark  0.725694  0.796429      5    Denmark   0.758359
8     Finland  0.731719  0.803289      5    Finland   0.760968
9      France  0.608262  0.833572     17     France   0.734027
10    Germany  0.657177  0.800733     16    Germany   0.708552
11     Greece  0.627180  0.774886     13     Greece   0.698658
12    Hungary  0.588055  0.836852      8    Hungary   0.655738
13     Israel  0.693537  0.803484      6     Israel   0.743458
14      Italy  0.586813  0.701000     21      Italy   0

In [121]:
def build_minmax_national_df(stats_df, year_label):
    df = stats_df.copy()
    # ensure needed columns exist
    for c in ('Country','min','max','OBS_VALUE'):
        if c not in df.columns:
            raise KeyError(f"expected column '{c}' in stats_df")
    rows = []
    for _, r in df.iterrows():
        rows.append({'Country': r['Country'], 'Year': year_label, 'Type': 'Worst Region', 'Value': r['min']})
        rows.append({'Country': r['Country'], 'Year': year_label, 'Type': 'Best Region',  'Value': r['max']})
        rows.append({'Country': r['Country'], 'Year': year_label, 'Type': 'National Average','Value': r['OBS_VALUE']})
    return pd.DataFrame(rows)

# prepare combined data
df_2024 = build_minmax_national_df(regional_stats_filtered_max, 'Circa 2024')
df_2010 = build_minmax_national_df(regional_stats_filtered_min, 'Circa 2010')
combined = pd.concat([df_2024, df_2010], ignore_index=True)

# compute sort order from 2024 gap (largest gaps first)
if 'max' in regional_stats_filtered_max.columns and 'min' in regional_stats_filtered_max.columns:
    regional_stats_filtered_max = regional_stats_filtered_max.assign(gap = regional_stats_filtered_max['max'] - regional_stats_filtered_max['min'])
    country_order = regional_stats_filtered_max.sort_values('gap', ascending=False)['Country'].tolist()
else:
    country_order = sorted(combined['Country'].unique())

# rules datasource: min/max per Country+Year (one row per country+year)
rules = combined[combined['Type'].isin(['Worst Region','Best Region'])].groupby(['Country','Year'], as_index=False).agg(min_val=('Value','min'), max_val=('Value','max'))

import altair as alt
alt.data_transformers.disable_max_rows()

# build rule from the same top-level data using transform_aggregate (min/max per Country+Year)
rule = alt.Chart(combined).transform_aggregate(
    min_val='min(Value)',
    max_val='max(Value)',
    groupby=['Country', 'Year']
).mark_rule(strokeWidth=12, opacity=0.35, color='lightgray').encode(
    x=alt.X('Country:N', sort=country_order, title='Country'),
    y=alt.Y('min_val:Q', title='HCI (composite)'),
    y2='max_val:Q',
    tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('min_val:Q', title='min'), alt.Tooltip('max_val:Q', title='max')]
)

points = alt.Chart(combined).mark_point(filled=True, size=120).encode(
    x=alt.X('Country:N', sort=country_order),
    y=alt.Y('Value:Q'),
    shape=alt.Shape('Type:N',
                    scale=alt.Scale(domain=['Best Region','National Average','Worst Region'],
                                    range=['triangle-up','square','triangle-down'])),
    color=alt.Color('Type:N',
                    scale=alt.Scale(domain=['Best Region','National Average','Worst Region'],
                                    range=['#3C3F3C','#4997536E','#3C3F3C'])),
    tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('Type:N'), alt.Tooltip('Value:Q', format='.3f')]
)

# layer charts (both use the same top-level data) then facet; pass data into facet
layered = alt.layer(rule, points).resolve_scale(color='independent', shape='independent').properties(
    width=20 * len(country_order[:20]),  # set width here
    height=360,
    title='Regional min/max (rule) and national value (point) — Circa 2010 vs Circa 2024'
)

chart = layered.facet(
    column=alt.Column('Year:N', header=alt.Header(labelOrient='bottom', title=None)),
    data=combined
).configure_title(anchor='start')

# show or save
# chart.save(r"C:\Users\lopez\github\capp30239\static_visualization_project\graphs\shape_hci_2010_2024.png")
chart

alt.FacetChart(...)

In [128]:
import altair as alt
alt.data_transformers.disable_max_rows()

# Filter for 2024 only
combined_2024 = combined[combined['Year'] == 'Circa 2024']

# Get country order by National Average (descending)
national_avg_2024 = combined_2024[combined_2024['Type'] == 'National Average']
country_order = national_avg_2024.sort_values('Value', ascending=False)['Country'].tolist()

# Min/max rule for 2024 (horizontal)
rule_2024 = alt.Chart(combined_2024).transform_aggregate(
    min_val='min(Value)',
    max_val='max(Value)',
    groupby=['Country']
).mark_rule(strokeWidth=10, opacity=0.4, color='gray').encode(
    y=alt.Y('Country:N', sort=country_order, title='Country'),
    x=alt.X('min_val:Q', title='HCI (composite)'),
    x2='max_val:Q',
    tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('min_val:Q', title='2024 min'), alt.Tooltip('max_val:Q', title='2024 max')]
)

# National average points for 2024 (horizontal)
points_2024 = alt.Chart(national_avg_2024).mark_point(
    filled=True, size=120, color='red'
).encode(
    y=alt.Y('Country:N', sort=country_order),
    x=alt.X('Value:Q'),
    shape=alt.Shape('Type:N', scale=alt.Scale(domain=['National Average'], range=['square'])),
    tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('Value:Q', format='.3f')]
)

# Combine layers
chart = alt.layer(
    rule_2024, points_2024
).properties(
    width=800,
    height=20 * len(country_order[:20]),
    title='Regional min/max (rule) and national value (point) — Circa 2024'
).configure_title(anchor='start')

# chart.save(r"C:\Users\lopez\github\capp30239\static_visualization_project\graphs\shape_hci_2024_horizontal.png")
chart

alt.LayerChart(...)

In [184]:
import altair as alt
alt.data_transformers.disable_max_rows()

# Filter for 2024 and 2010 only
combined_2024 = combined[combined['Year'] == 'Circa 2024']
combined_2010 = combined[combined['Year'] == 'Circa 2010']

# Fix Colombia value if needed
combined_2010.loc[
    (combined_2010["Country"] == "Colombia") & (combined_2010["Type"] == "National Average"),
    "Value"
] = 0.54

# Add a new column to distinguish year and type
national_avg_2024 = combined_2024[combined_2024['Type'] == 'National Average'].copy()
national_avg_2024['YearType'] = '2024 National Average'
national_avg_2010 = combined_2010[combined_2010['Type'] == 'National Average'].copy()
national_avg_2010['YearType'] = '2010 National Average'

# Combine for plotting
combined_points = pd.concat([national_avg_2010, national_avg_2024], ignore_index=True)

# Get country order by 2024 National Average (descending)
country_order = national_avg_2024.sort_values('Value', ascending=False)['Country'].tolist()

# Draw lines connecting 2010 to 2024 for each country
lines = alt.Chart(combined_points).mark_line(
    color="#9B98982B",
    strokeWidth=5
).encode(
    y=alt.Y('Country:N', sort=country_order),
    x=alt.X('Value:Q', scale=alt.Scale(domain=[0.5, 0.9]), title='HCI (composite)'),
    detail='Country:N'
)

# Plot points with shape and color by YearType
points = alt.Chart(combined_points).mark_point(
    filled=True,
    stroke='black',        # border color
    strokeWidth=1
).encode(
    y=alt.Y('Country:N', sort=country_order,  axis=alt.Axis(grid=True, gridColor="#e0e0e055")),
    x=alt.X('Value:Q', scale=alt.Scale(domain=[0.5, 0.9]), title='HCI (composite)', axis=alt.Axis(grid=True, gridColor="#e0e0e055")),
    shape=alt.Shape('YearType:N', scale=alt.Scale(domain=['2010 National Average', '2024 National Average'],
                                                  range=['square', 'triangle-right']),
                    legend=alt.Legend(title='')),
    color=alt.Color('YearType:N', scale=alt.Scale(domain=['2010 National Average', '2024 National Average'],
                                                range=["#474646FF", "#FFFF00"]),
                                                legend=alt.Legend(title='', orient='top')),
    size=alt.Size('YearType:N',
                  scale=alt.Scale(domain=['2010 National Average', '2024 National Average'],
                                  range=[60, 100])),  # 60 for square, 100 for triangle
    tooltip=[alt.Tooltip('Country:N'), alt.Tooltip('Value:Q', format='.3f'), alt.Tooltip('YearType:N')]
)

chart = alt.layer(
    lines, points
).properties(
    width=600,
    height=20 * len(country_order[:20]),
    title='Human Capital Index (HCI), Circa 2010 - Circa 2024'
).configure_title(anchor='start')

chart
# save this graph as png
chart.save(r"C:\Users\lopez\github\capp30239\static_visualization_project\graphs\Line_hci_2010_2024.png")

In [123]:
print(combined.head(30))

      Country        Year              Type     Value
0   Australia  Circa 2024      Worst Region  0.715821
1   Australia  Circa 2024       Best Region  0.882931
2   Australia  Circa 2024  National Average  0.786343
3     Austria  Circa 2024      Worst Region  0.678598
4     Austria  Circa 2024       Best Region  0.783333
5     Austria  Circa 2024  National Average  0.713165
6    Bulgaria  Circa 2024      Worst Region  0.572023
7    Bulgaria  Circa 2024       Best Region  0.752261
8    Bulgaria  Circa 2024  National Average  0.639161
9      Canada  Circa 2024      Worst Region  0.660886
10     Canada  Circa 2024       Best Region  0.849759
11     Canada  Circa 2024  National Average  0.787449
12      Chile  Circa 2024      Worst Region  0.651181
13      Chile  Circa 2024       Best Region  0.740453
14      Chile  Circa 2024  National Average  0.688405
15   Colombia  Circa 2024      Worst Region  0.491184
16   Colombia  Circa 2024       Best Region  0.712793
17   Colombia  Circa 2024  N

In [ ]:
def create_national_data(countries_df):
    """Prepare national average data for chart."""
    national_data = countries_df.copy()
    national_data['Type'] = 'National Average'
    national_data['Reference area'] = national_data['Reference area']
    return national_data[['Reference area', 'OBS_VALUE', 'Type']]

def combine_all_points(minmax_points, national_data):
    """Combine min/max points and national average data."""
    return pd.concat([minmax_points, national_data], ignore_index=True)



country_sort_order = get_country_sort_order(countries_filtered)
national_data_max = create_national_data(regional_data_total_max)
all_points_data = combine_all_points(minmax_points, national_data_max)

In [ ]:

def create_regional_gap_chart_year(chart_df, country_order, year):
    """
    Creates a chart showing the gap between best and worst regions for each country.
    """
    chart_df_year = chart_df[chart_df['which'] == year]
    gap_bars = alt.Chart(chart_df_year[chart_df_year['Type'].isin(['Worst Region', 'Best Region'])]).mark_rule(
        strokeWidth=8,
        stroke='lightgray',
        opacity=0.4
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country'),
        y=alt.Y('min(Value):Q', axis=alt.Axis(gridColor="#f3f1f1e1")),
        y2=alt.Y2('max(Value):Q'),
        tooltip=[
            alt.Tooltip('Country:N', title='Country'),
            alt.Tooltip('min(Value):Q', title='Worst Region Deviation'),
            alt.Tooltip('max(Value):Q', title='Best Region Deviation')
        ]
    )
    return gap_bars

def create_deviation_points_chart(chart_df, country_order, average_color):
    """
    Creates a points chart for deviations from regional median.
    """
    points = alt.Chart(chart_df).mark_point(
        size=100,
        filled=True
    ).encode(
        x=alt.X('Country:N', sort=country_order, title='Country'),
        y=alt.Y('Value:Q', title='Deviation from Regional Median (percentage points)'),
        shape=alt.Shape('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['triangle-up', 'square', 'triangle-down']),
            legend=alt.Legend(orient='top'),
            title=''),
        color=alt.Color('Type:N',
            scale=alt.Scale(domain=['Best Region', 'National Average', 'Worst Region'],
                            range=['#3C3F3C', average_color, "#3C3F3C"]),
            legend=alt.Legend(orient='top'),
            title=''),
        tooltip=['Country', 'Type', 'Value']
    )
    return points

In [54]:
def create_regional_variation_chart(chart_df, country_order, average_color, chart_title= ""):
    """
    Combines gap bars, zero line, and deviation points into a single chart.
    """
    gap_bars = create_regional_gap_chart(chart_df, country_order)
    zero_line = create_zero_line()
    points = create_deviation_points_chart(chart_df, country_order, average_color)
    main_chart = (gap_bars + zero_line + points).properties(
        title=alt.TitleParams(
            text=chart_title,
            anchor='start',
            offset=10,
            dx=20
        ),
        width=800,
        height=400
    ).resolve_legend(
        color='independent',
        shape='independent'
    )
    source_text = create_source_text()
    chart = alt.vconcat(
        main_chart,
        source_text,
        spacing=10
    ).resolve_scale(
        color='independent',
        shape='independent'
    )
    return chart

In [55]:
create_regional_variation_chart(hci_change_df, country_order, "#4997536E", chart_title= "HCI Change by Country (2010 - 2024)")

KeyError: 'Type'

In [421]:
# put together the scatter1 - scatter4 into a single image two up two down

chart_gdp_combined = alt.hconcat(
    alt.vconcat(
        chart_gdp1,
        chart_gdp3
    ),
    alt.vconcat(
        chart_gdp2,
        chart_gdp4
    )
).resolve_scale(
    color='independent',
    shape='independent'
)

In [423]:
chart_gdp_combined.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\scatter_gdp_combined.png")
chart_gdp_combined

alt.HConcatChart(...)

In [256]:
print("Life Expectancy Data:", Life)
print("Survival Rate Data:", survival)
print("Attainment Data:", attainment)
print("Enrollment Data:", enrollment)

Life Expectancy Data:      Reference area  OBS_VALUE              Type
3         Australia      83.00  National Average
6           Austria      81.90  National Average
12         Bulgaria      75.80  National Average
18           Canada      81.34  National Average
21      Switzerland      84.30  National Average
24            Chile      81.00  National Average
30         Colombia      76.80  National Average
39          Czechia      79.90  National Average
42          Germany      81.10  National Average
45          Denmark      81.80  National Average
48            Spain      83.77  National Average
54          Finland      81.60  National Average
57           France      83.00  National Average
62   United Kingdom      80.70  National Average
63           Greece      81.80  National Average
69          Hungary      76.54  National Average
75            India      69.90  National Average
84           Israel      82.60  National Average
87            Italy      83.40  National Averag